In [2]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from tile_coder import TileCoder
from tqdm import tqdm

In [18]:
env = gym.make("MountainCar-v0", max_episode_steps=5000)
print(env.observation_space)
coder = TileCoder(
    low=[-1.2, -0.07],
    high=[0.6, 0.07],
    num_tiles=8,
    tile_per_dimension=[8, 8],
    num_actions=3
)

Box([-1.2  -0.07], [0.6  0.07], (2,), float32)


In [19]:
def epsilon_greedy(w:np.ndarray, state, coder:TileCoder, epsilon:0.1):
    if np.random.rand() < epsilon:
        return np.random.choice(coder.num_actions)

    values = []
    for i in range(coder.num_actions):
        values.append(w.T@coder.get_vector(state, i))

    return int(np.argmax(values))


In [20]:
trace = "accumulating" # or replacing

def sarsa_lambda_binary_features(
    w:np.ndarray,
    env:gym.Env,
    coder:TileCoder,
    episodes:int,
    lambda_:float=0.4,
    gamma:float=0.99,
    alpha:float=0.1,
    epsilon:float=0.1,
    trace:str='accumulating'
):
    assert trace in ['replacing', 'accumulating'], 'Unknown trace strategy'
    rewards_per_episode = []
    for episode in tqdm(range(episodes)):
        state = env.reset()[0]
        action = epsilon_greedy(w, state, coder, epsilon)
        x = coder.get_vector(state, action)
        z = 0
        terminated = False
        total_rewards = 0
        while not terminated:
            next_state, reward, terminated, truncated, _ = env.step(action)
            terminated = terminated or truncated
            total_rewards += reward # for logging
            delta = reward - w.T@x
            if trace == 'accumulating':
                z = z + x
            else:
                z = np.maximum(z, x)

            if terminated:
                w = w + alpha * delta * z
                break

            next_action = epsilon_greedy(w, next_state, coder, epsilon)
            x_prime = coder.get_vector(next_state, next_action)
            delta += gamma * w.T@x_prime
            w = w + alpha * delta * z
            z = lambda_ * gamma * z
            state = next_state
            action = next_action
            x = x_prime
        rewards_per_episode.append(total_rewards)
    return w, rewards_per_episode

In [21]:
runs = 5
alphas = np.arange(0.2, 1.0, 0.2)
results = []
for alpha in alphas:
    mean_rewards = 0
    for run in range(runs):
        w = np.zeros((coder.num_features,), dtype=np.float32)
        w, rewards_per_episode = sarsa_lambda_binary_features(
            w=w,
            env=env,
            coder=coder,
            episodes=20,
            alpha=alpha / 8,
            lambda_=0.9
        )
        mean_rewards += np.mean(rewards_per_episode)
    results.append(mean_rewards / runs)

100%|██████████| 20/20 [00:40<00:00,  2.02s/it]


In [10]:
# run infer
from gymnasium.wrappers import RecordVideo
eval_env = gym.make("MountainCar-v0", render_mode='rgb_array')
eval_env = RecordVideo(
    eval_env,
    "videos/",
    episode_trigger=lambda eps: True
)
state = eval_env.reset()[0]
terminated = False
steps = 0

while not terminated:
    print(f"\rStep {steps+1:<5}", end="")
    action = epsilon_greedy(w, state, coder, 0.0)
    state, _, terminated, _, _ = eval_env.step(action)
    steps += 1

print("agent took", steps, "steps")
eval_env.close()

/home/alireza/miniconda3/envs/torch/lib/python3.13/site-packages/gymnasium/wrappers/rendering.py:283: UserWarning: WARN: Overwriting existing videos at /home/alireza/Desktop/Reinforcement_Learning/Chapter 12/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Step 185  agent took 185 steps
